# บทเรียนที่ 05 - Agentic RAG


## การตั้งค่า

โน้ตบุ๊กนี้สาธิตรูปแบบ Agentic RAG (Retrieval-Augmented Generation) โดยใช้ Microsoft Agent Framework

**สิ่งที่ต้องเตรียม:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — จุดสิ้นสุดบริการ Azure AI Search ของคุณ
- `AZURE_SEARCH_API_KEY` — คีย์ API ของ Azure AI Search ของคุณ
- การปรับใช้ Azure OpenAI ที่กำหนดค่าผ่านตัวแปรสภาพแวดล้อม
- เข้าสู่ระบบ Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Agentic RAG คืออะไร?

RAG แบบดั้งเดิมจะทำตามขั้นตอนคงที่: ดึงเอกสารมาแล้วจึงสร้างคำตอบ **Agentic RAG** ก้าวไปไกลกว่านั้นโดยให้เอเยนต์มีอิสระในการตัดสินใจ **เมื่อใด** และ **อย่างไร** ในการดึงข้อมูล

ด้วย Agentic RAG เอเยนต์สามารถ:
- **ตัดสินใจ** ว่าจำเป็นต้องดึงข้อมูลก่อนตอบคำถามหรือไม่
- **เลือก** แหล่งข้อมูลหรือเครื่องมือที่จะสอบถาม
- **ประเมินผล** ของผลลัพธ์ที่ดึงมาและดำเนินการดึงข้อมูลเพิ่มเติมหากการพยามครั้งแรกยังไม่เพียงพอ
- **รวม** ข้อมูลจากหลายขั้นตอนการดึงให้เป็นคำตอบที่สอดคล้องกัน

สิ่งนี้ทำให้เอเยนต์มีความยืดหยุ่นและแม่นยำมากกว่าการทำงานแบบดึงแล้วสร้างคำตอบแบบคงที่


## การสร้างเครื่องมือค้นหา

ใน Agentic RAG แหล่งข้อมูลภายนอกจะถูกห่อหุ้มเป็น **เครื่องมือ** ที่เอเจนต์สามารถเรียกใช้ได้ตามต้องการ ซึ่งช่วยให้เอเจนต์มองว่าการดึงข้อมูลเป็นเพียงอีกหนึ่งการกระทำที่สามารถทำได้ ไม่ใช่ขั้นตอนที่บังคับ

ด้านล่างนี้เราได้กำหนดฐานความรู้เกี่ยวกับการท่องเที่ยวและเปิดเผยมันเป็นเครื่องมือที่เอเจนต์สามารถเรียกใช้เพื่อค้นหาข้อมูลปลายทาง


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## การสร้างตัวแทน RAG

ขณะนี้เราจะสร้างตัวแทนที่ถูกสั่งให้ **ดึงข้อมูลเสมอก่อนตอบคำถาม** ตัวแทนจะใช้เครื่องมือ `search_travel_knowledge` เพื่อยึดคำตอบของตนกับฐานความรู้แทนที่จะพึ่งพาข้อมูลการฝึกสอนของตัวเอง


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## การดึงข้อมูลแบบวนซ้ำ — รูปแบบผู้สร้าง-ผู้ตรวจสอบ

ข้อได้เปรียบสำคัญของ Agentic RAG คือ **การดึงข้อมูลแบบวนซ้ำ** ตัวแทนสามารถทำการค้นหาหลายรอบเพื่อยืนยัน ปรับปรุง หรือต่อยอดจากข้อมูลเบื้องต้น — คล้ายกับเวิร์กโฟลว์ "ผู้สร้าง-ผู้ตรวจสอบ":

1. **ขั้นตอนผู้สร้าง**: ตัวแทนดึงข้อมูลเบื้องต้นและร่างคำตอบ
2. **ขั้นตอนผู้ตรวจสอบ**: ตัวแทนทำการดึงข้อมูลเพิ่มเติมเพื่อยืนยันรายละเอียดหรือเติมเต็มข้อมูลที่ขาด

ด้านล่าง ตัวแทนถูกถามคำถามที่ต้องเปรียบเทียบหลายจุดหมายปลายทาง ซึ่งกระตุ้นให้ค้นหาหลายครั้ง


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีสร้างระบบ **Agentic RAG** โดยใช้ Microsoft Agent Framework:

- **Agentic RAG** ช่วยให้เอเจนต์ตัดสินใจด้วยตนเองเมื่อจะดึงข้อมูล ทำให้การดึงข้อมูลมีความไดนามิกแทนที่จะเป็นแบบคงที่
- **เครื่องมือเป็นแหล่งข้อมูล**: ฐานความรู้ภายนอก (เช่น Azure AI Search) ถูกห่อหุ้มเป็นเครื่องมือที่เอเจนต์สามารถเรียกใช้ได้
- **การดึงข้อมูลแบบทำซ้ำ**: รูปแบบ maker-checker ช่วยให้เอเจนต์ดำเนินการดึงข้อมูลซ้ำหลายรอบ — ค้นหา ตรวจสอบ และปรับปรุง — ก่อนออกคำตอบสุดท้าย

ในการใช้งานจริง คุณจะต้องแทนที่ `TRAVEL_KNOWLEDGE_BASE` ในหน่วยความจำด้วยดัชนี Azure AI Search จริงเพื่อจัดการกับการดึงเอกสารท่องเที่ยวในปริมาณมาก


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
